<a href="https://colab.research.google.com/github/ReposofPriyanka/flyrank-ai-internship-ml-track/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit


###**Research paper findings and methodology questions**



### Finding 1 — Content freshness and performance

**Paper:** FlyRank — The State of AI-Driven SEO, March 2026

**Finding:** The report identifies 31–90 days as the strongest stable freshness window, with a 7.88:1 growth-to-decline ratio. It also reports that 365+ day content refreshed within the previous 30 days had a higher average health score and impressions than older content that had not been refreshed recently.

**Evidence:** The report describes a 3.2× health-score difference (10.7 to 34.5) and 57× impressions difference (71 to 4,039). It cautions that the 361+ day freshness bucket was very small and unstable.

**Methodology question:** Were refreshed pages compared with similar unrefreshed pages matched on prior impressions, age, and content quality? If not, how might existing differences between the groups affect the observed association?



### Finding 2 — Model feature importance and health score

**Paper:** FlyRank — The State of AI-Driven SEO, March 2026

**Finding:** In the exploratory Random Forest analysis, average position had the highest reported feature importance (43%), followed by impressions (32%) and scroll depth (15%) when predicting the FlyRank health score.

**Evidence:** The report notes that the health score is partly constructed from inputs such as position and impressions. Therefore, feature importance is descriptive and does not establish external causation.

**Methodology question:** Since average position and impressions are already components of the health score, how would the feature-importance results change if the model predicted an outcome independent of those inputs?

### Setup & Preparation

In [1]:
# Setup, March 2026 page-level data, and two signal checks

!pip -q install duckdb pandas pyarrow

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from pathlib import Path

# -------------------------
# 1. Connect to FlyRank data
# -------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is missing. Add your Hugging Face READ token "
        "in Colab Secrets and enable notebook access."
    )

con = duckdb.connect()

# Escape quotes in the token before creating the secret.
safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# -------------------------
# 2. Load the March 2026 slice
# -------------------------

# One row per client-content pair after aggregation.
# Only March 2026 data is used.
# GSC availability must explicitly be TRUE.

query = f"""
WITH march_daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        gsc_avg_position
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
),
page_month AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) AS sum_position,
        COUNT(*) AS observed_days
    FROM march_daily
    GROUP BY
        client_hash_id,
        content_hash_id
)
SELECT
    client_hash_id,
    content_hash_id,
    impressions,
    clicks,
    sum_position,
    observed_days,
    CASE
        WHEN impressions > 0
        THEN clicks * 100.0 / impressions
        ELSE NULL
    END AS ctr,
    CASE
        WHEN impressions > 0
        THEN sum_position * 1.0 / impressions
        ELSE NULL
    END AS avg_position
FROM page_month
WHERE impressions > 0
"""

df = con.sql(query).df()

if df.empty:
    raise ValueError(
        "No March 2026 rows were returned. Check warehouse access "
        "and the month partition."
    )

# Keep only rows with usable position data for this rule.
df["avg_position"] = pd.to_numeric(
    df["avg_position"], errors="coerce"
)

df["ctr"] = pd.to_numeric(
    df["ctr"], errors="coerce"
)

df = df.replace([np.inf, -np.inf], np.nan)

df = df.dropna(
    subset=["impressions", "ctr", "avg_position"]
).copy()

df = df[df["avg_position"] > 0].copy()

# Stable internal row identifier for this notebook.
df = df.reset_index(drop=True)
df["row_id"] = np.arange(1, len(df) + 1)

print("March 2026 page-level dataset")
print("Rows:", len(df))
print("Unique content items:", df["content_hash_id"].nunique())
print("Unique clients:", df["client_hash_id"].nunique())
print("Total impressions:", int(df["impressions"].sum()))

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 page-level dataset
Rows: 175304
Unique content items: 175304
Unique clients: 47
Total impressions: 280655033


,client_hash_id,content_hash_id,impressions,clicks,sum_position,observed_days,ctr,avg_position,row_id
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,44965.0,31,0.107313,6.893301,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,1456.0,31,0.000000,3.214128,2
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,36794.0,31,0.106572,6.535346,3
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,36762.0,31,0.262945,7.435680,4
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,1661.0,31,0.233100,3.871795,5


## 2. My model under an honest split (before/after)

#### Model Data Preparation

In [2]:
# Build the modeling dataset
# March features + April outcome

query_model = f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) AS sum_position
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month IN ('2026-03', '2026-04')
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id,
        month
),

monthly_metrics AS (
    SELECT
        client_hash_id,
        content_hash_id,
        month,
        impressions,
        clicks,

        CASE
            WHEN impressions > 0
            THEN sum_position * 1.0 / impressions
            ELSE NULL
        END AS avg_position

    FROM monthly
)

SELECT
    march.client_hash_id,
    march.content_hash_id,

    march.impressions,
    march.clicks,
    march.avg_position,

    april.impressions AS april_impressions,

    (
        (april.impressions - march.impressions)
        * 100.0 / march.impressions
    ) AS future_trend_pct

FROM monthly_metrics AS march

INNER JOIN monthly_metrics AS april
    ON march.client_hash_id = april.client_hash_id
    AND march.content_hash_id = april.content_hash_id

WHERE march.month = '2026-03'
  AND april.month = '2026-04'
  AND march.impressions > 0
"""

model_frame = con.sql(query_model).df()

# Clean invalid values
model_frame = model_frame.replace(
    [np.inf, -np.inf],
    np.nan
)

model_frame = model_frame.dropna(
    subset=[
        "impressions",
        "clicks",
        "avg_position",
        "future_trend_pct"
    ]
).copy()

# Keep rows with valid position data
model_frame = model_frame[
    model_frame["avg_position"] > 0
].copy()

model_frame = model_frame.reset_index(drop=True)

print("Modeling dataset created")
print("Rows:", len(model_frame))
print("Columns:", model_frame.columns.tolist())
print("Missing values:")
display(model_frame.isna().sum())

display(model_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling dataset created
Rows: 157790
Columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'april_impressions', 'future_trend_pct']
Missing values:


,0
client_hash_id,0
content_hash_id,0
impressions,0
clicks,0
avg_position,0
april_impressions,0
future_trend_pct,0


,client_hash_id,content_hash_id,impressions,clicks,avg_position,april_impressions,future_trend_pct
0,client_62f4a7e64f5e0096,content_f1c085e5ea530266,207.0,0.0,5.449275,64.0,-69.082126
1,client_62f4a7e64f5e0096,content_82f4df1a7531638c,4.0,0.0,6.500000,15.0,275.000000
2,client_62f4a7e64f5e0096,content_c4901b51a12b1c3b,218.0,1.0,7.793578,81.0,-62.844037
3,client_62f4a7e64f5e0096,content_ecb932abf0674642,41.0,0.0,15.439024,31.0,-24.390244
4,client_62f4a7e64f5e0096,content_64706c8afebebb8c,666.0,2.0,5.243243,131.0,-80.330330


#### Validating the modeling dataset

In [3]:
print("Dataset shape:", model_frame.shape)

print("\nFeature columns:")
feature_columns = [
    "impressions",
    "clicks",
    "avg_position"
]

print(feature_columns)

print("\nTarget:")
print("future_trend_pct")

print("\nTarget summary:")
display(model_frame["future_trend_pct"].describe())

print("\nDuplicate client-content pairs:")
print(
    model_frame.duplicated(
        subset=["client_hash_id", "content_hash_id"]
    ).sum()
)

print("\nMissing values:")
display(model_frame.isna().sum())

assert not model_frame.empty, "Modeling dataset is empty."

assert model_frame[
    ["client_hash_id", "content_hash_id"]
].duplicated().sum() == 0, "Duplicate page pairs found."

assert model_frame["future_trend_pct"].notna().all()

Dataset shape: (157790, 7)

Feature columns:
['impressions', 'clicks', 'avg_position']

Target:
future_trend_pct

Target summary:


,future_trend_pct
count,157790.000000
mean,167.945856
std,3133.456470
min,-99.980522
25%,-50.472444
50%,-17.014032
75%,40.212268
max,540400.000000



Duplicate client-content pairs:
0

Missing values:


,0
client_hash_id,0
content_hash_id,0
impressions,0
clicks,0
avg_position,0
april_impressions,0
future_trend_pct,0


In [4]:
# ML-09 - Section 2
# Compare the original random split with a client-grouped split.

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

# Use the actual three features from Week-5.
feature_columns = [
    "impressions",
    "clicks",
    "avg_position"
]

target_col = "future_trend_pct"
group_col = "client_hash_id"

# Keep only required columns and remove invalid rows.
audit_df = model_frame[
    feature_columns + [target_col, group_col]
].copy()

audit_df = audit_df.replace(
    [np.inf, -np.inf], np.nan
)

audit_df = audit_df.dropna(
    subset=feature_columns + [target_col, group_col]
).copy()

X = audit_df[feature_columns]
y = audit_df[target_col]
groups = audit_df[group_col]

# Confirm the number of clients and rows.
print("Rows:", len(audit_df))
print("Unique clients:", groups.nunique())

# Create a client-grouped 80/20 split.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

# Verify that no client appears in both sets.
assert train_clients.isdisjoint(test_clients)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

# Fit the same Random Forest configuration.
grouped_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

grouped_model.fit(X_train, y_train)

grouped_prediction = grouped_model.predict(X_test)

# Baseline: training-set median.
baseline_prediction = np.full(
    len(y_test),
    y_train.median()
)

def evaluate(name, actual, predicted):
    return {
        "Evaluation": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": r2_score(actual, predicted)
    }

grouped_results = pd.DataFrame([
    evaluate(
        "Grouped median baseline",
        y_test,
        baseline_prediction
    ),
    evaluate(
        "Grouped Random Forest",
        y_test,
        grouped_prediction
    )
])

display(grouped_results.round(3))


Rows: 157790
Unique clients: 46
Training rows: 136739
Test rows: 21051
Training clients: 36
Test clients: 10
Client overlap: 0


,Evaluation,MAE,RMSE,R2
0,Grouped median baseline,391.769,7413.032,-0.002
1,Grouped Random Forest,420.688,7381.408,0.006


##### **Grouped validation results**

I re-evaluated the Random Forest model using a client-grouped 80/20 train-test split. The dataset contained 157,790 rows across 46 clients. The training set contained 136,739 rows from 36 clients, while the test set contained 21,051 rows from 10 unseen clients. There was no client overlap between the two sets.

The grouped Random Forest achieved an MAE of 420.688, RMSE of 7,381.408, and R² of 0.006. The grouped median baseline achieved an MAE of 391.769, RMSE of 7,413.032, and R² of -0.002.

The Random Forest had a higher MAE than the baseline, although its RMSE was slightly lower. Both R² values were close to zero, indicating limited measured predictive value under this evaluation.

This grouped evaluation measures performance on unseen clients within the available March-to-April 2026 dataset. It does not establish performance on future months. The results support using the model only as an exploratory decision-support candidate, not as a reliable predictor of future impression changes.


### **Random-split vs. grouped validation**

In the Week 5 random-split evaluation, the Transformed Random Forest achieved an MAE of 214.248, RMSE of 2,908.842, and R² of −0.0018. The median baseline achieved an MAE of 215.140, RMSE of 2,911.742, and R² of −0.0038. The model’s errors were slightly lower than the baseline’s, while both R² values remained close to zero.

In the ML-09 client-grouped evaluation, the Random Forest achieved an MAE of 420.688, RMSE of 7,381.408, and R² of 0.006. The grouped median baseline achieved an MAE of 391.769, RMSE of 7,413.032, and R² of −0.002. The Random Forest had higher MAE but slightly lower RMSE than the baseline.

 These results should not be interpreted as a direct apples-to-apples comparison: the Week 5 table reports a transformed model under a random split, while ML-09 evaluates a Random Forest under a client-grouped split. The grouped evaluation tests performance on clients excluded from training and provides a stricter check of generalization across clients. Both evaluations show R² values near zero, so the evidence does not support claiming strong predictive performance.

## 3. Leakage audit


In [5]:
# Columns that must not be used as predictors because they
# reveal the label, future outcome, or record identity.

forbidden_terms = [
    "future",
    "label",
    "trend_pct",
    "trend_direction",
    "april",
    "target",
    "client_id",
    "content_id",
    "url",
]

suspicious_columns = [
    col for col in model_frame.columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("Potentially sensitive or leakage-related columns:")
print(suspicious_columns)

print("\nSelected features:")
print(feature_columns)

overlap = [
    col for col in feature_columns
    if col in suspicious_columns
]

if overlap:
    print("\nWARNING — review these selected features:", overlap)
else:
    print("\nNo selected feature matches the known leakage-risk terms.")

Potentially sensitive or leakage-related columns:
['april_impressions', 'future_trend_pct']

Selected features:
['impressions', 'clicks', 'avg_position']

No selected feature matches the known leakage-risk terms.


The name-based leakage screening flagged april_impressions and future_trend_pct. Neither field is included in the model predictors. The selected features—impressions, clicks, and avg_position—did not match the known leakage-risk terms. This check is a preliminary screen, not proof that the features are leakage-free. I also reviewed their timing and construction to confirm that the predictors represent March-period information and that April outcome data is used only to calculate the target.

In [6]:
# Feature-level review based on the known Week 5 data construction.

feature_audit = [
    {
        "feature": "impressions",
        "review": "March-period value; available before the April outcome.",
        "risk": "Part of the target's percentage-change denominator, so its structural relationship to the target must be disclosed."
    },
    {
        "feature": "clicks",
        "review": "March-period value; available before the April outcome.",
        "risk": "No direct target component identified; confirm it is not calculated using April data."
    },
    {
        "feature": "avg_position",
        "review": "March-period value; available before the April outcome.",
        "risk": "Confirm it is not derived from future-period information."
    },
    {
        "feature": "future_trend_pct",
        "review": "April-versus-March outcome used as the target.",
        "risk": "Must never be included as a predictor."
    },
    {
        "feature": "client_id / content_id",
        "review": "Identifiers used for grouping or joining records.",
        "risk": "Must not be used as model predictors."
    },
    {
        "feature": "trend_pct / trend_direction",
        "review": "Derived trend fields associated with the label.",
        "risk": "Must not be used as predictors because they can reveal the target."
    },
]

import pandas as pd

display(pd.DataFrame(feature_audit))

,feature,review,risk
0,impressions,March-period value; available before the April...,Part of the target's percentage-change denomin...
1,clicks,March-period value; available before the April...,No direct target component identified; confirm...
2,avg_position,March-period value; available before the April...,Confirm it is not derived from future-period i...
3,future_trend_pct,April-versus-March outcome used as the target.,Must never be included as a predictor.
4,client_id / content_id,Identifiers used for grouping or joining records.,Must not be used as model predictors.
5,trend_pct / trend_direction,Derived trend fields associated with the label.,Must not be used as predictors because they ca...


#### **Leakage audit findings**

I reviewed the model inputs against the target construction and the known dataset leakage risks. The target, future_trend_pct, represents the change in impressions from March to April. The model uses only March-period impressions, clicks, and avg_position as predictors.

The name-based screening flagged april_impressions and future_trend_pct; neither is included in the model predictors. I also excluded identifiers such as client and content IDs because they are used for grouping or joining, not as predictive inputs. Derived trend fields such as trend_pct and trend_direction were excluded because they are associated with the label and could reveal information about the outcome.

March impressions are available before the April outcome, but they also form the denominator of the percentage-change target. This creates a structural relationship with the target that should be considered when interpreting model performance. April outcome information is used only to calculate the target, not as a predictor.

This audit reduces known leakage risks, but it does not prove that every source field is leakage-free. The review depends on the documented timing and construction of the underlying data.

## 4. Claim rewrite


In [7]:
# Research claim audit

revised_claim = """
In the client-grouped evaluation, the Random Forest showed limited
predictive value for estimating April impression changes from
March-period features. Its MAE was higher than the median baseline,
while its RMSE was slightly lower; both models had R² values close
to zero. These results do not establish reliable predictive
performance on unseen clients or future months. The model may be
explored as a decision-support candidate, but its outputs should
not be treated as dependable forecasts.
"""

claim_evidence = {
    "evaluation": "Client-grouped holdout",
    "random_forest_MAE": 420.688,
    "median_baseline_MAE": 391.769,
    "random_forest_RMSE": 7381.408,
    "median_baseline_RMSE": 7413.032,
    "random_forest_R2": 0.006,
    "median_baseline_R2": -0.002,
}

print(revised_claim)
print("\nSupporting evidence:")
for metric, value in claim_evidence.items():
    print(f"{metric}: {value}")



In the client-grouped evaluation, the Random Forest showed limited
predictive value for estimating April impression changes from
March-period features. Its MAE was higher than the median baseline,
while its RMSE was slightly lower; both models had R² values close
to zero. These results do not establish reliable predictive
performance on unseen clients or future months. The model may be
explored as a decision-support candidate, but its outputs should
not be treated as dependable forecasts.


Supporting evidence:
evaluation: Client-grouped holdout
random_forest_MAE: 420.688
median_baseline_MAE: 391.769
random_forest_RMSE: 7381.408
median_baseline_RMSE: 7413.032
random_forest_R2: 0.006
median_baseline_R2: -0.002


I revised the claim to reflect the grouped validation results rather than presenting the model as a reliable predictor. The evaluation measures performance on unseen clients within the available March-to-April 2026 dataset. It does not establish generalization to future months, and the near-zero R² values indicate that the model explains very little variation in the target under this evaluation.


A next step is to evaluate the same model configuration using a time-based holdout on a later period, once an appropriate development and test window is available. This would help assess whether the model’s performance carries forward to future observations. Any further claim should also be compared against a simple baseline and reviewed for stability across clients.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.